# 指令微调和监督微调

一个基础模型预测下一个词元。这就是全部，基础模型不会遵循指令，回答问题，或者拒绝有害的请求。监督微调就是一个词元预测期和有用的助手之间的桥梁。

## 基本概念

### 监督微调在干的事

监督微调的训练循环与预训练一样，但是**训练的语料**是不一样的。你在结构化的对话文本上训练而不是裸的文本。

```
{
    "system": "balabala",
    "user": "balabala",
    "assitant": "response sample."
}
```

# 动手构建


## 0. 环境准备

- 基础模型：本地 `models/gpt2`（HuggingFace `gpt2`, ~124M）
- 训练语料：`data/alpaca_sft_small.json`（100 条 Alpaca 格式）
- 下面单元格负责加载、预览，并把样本格式化成可训练的 prompt


In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path(".").resolve()
MODEL_DIR = ROOT / "models" / "gpt2"
DATA_PATH = ROOT / "data" / "alpaca_sft_small.json"

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
print("model dir exists:", MODEL_DIR.is_dir(), "->", MODEL_DIR)
print("data exists:", DATA_PATH.is_file(), "->", DATA_PATH)


In [ ]:
# 加载基础模型 GPT-2（优先本地；没有则从 HuggingFace 拉取并缓存到 models/gpt2）
if (MODEL_DIR / "config.json").exists():
    model_name_or_path = str(MODEL_DIR)
    print("Loading local GPT-2 from", model_name_or_path)
else:
    model_name_or_path = "gpt2"
    print("Local weights missing; downloading gpt2 ...")

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to(DEVICE)

# GPT-2 默认没有 pad_token；SFT/batch 时常用 eos 充当 pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# 若是首次下载，落盘方便离线复用
if model_name_or_path == "gpt2":
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    tokenizer.save_pretrained(MODEL_DIR)
    model.save_pretrained(MODEL_DIR)
    print("Saved to", MODEL_DIR)

n_params = sum(p.numel() for p in model.parameters())
print(f"GPT-2 ready | params={n_params/1e6:.1f}M | vocab={tokenizer.vocab_size}")


In [ ]:
# 加载 Alpaca 风格监督微调语料
with open(DATA_PATH, "r", encoding="utf-8") as f:
    sft_data = json.load(f)

print(f"samples: {len(sft_data)}")
print("keys:", sorted(sft_data[0].keys()))
print("--- example[0] ---")
print(json.dumps(sft_data[0], indent=2, ensure_ascii=False))


In [ ]:
# Alpaca prompt 模板（instruction/input -> 模型应续写 output）
ALPACA_TMPL = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

def format_alpaca(example: dict) -> dict:
    instruction = example["instruction"]
    inp = example.get("input") or ""
    output = example["output"]
    if not inp.strip():
        # 无额外 input 时仍保留空 Input 段，格式与常见 Alpaca 一致
        prompt = ALPACA_TMPL.format(instruction=instruction, input="")
    else:
        prompt = ALPACA_TMPL.format(instruction=instruction, input=inp)
    # 训练时通常把 prompt+response 拼在一起；后续会对 prompt 部分做 loss mask
    text = prompt + output + tokenizer.eos_token
    return {"prompt": prompt, "text": text, "response": output}

formatted = [format_alpaca(ex) for ex in sft_data]
print("--- formatted prompt preview ---")
print(formatted[0]["prompt"])
print(formatted[0]["response"])
print("--- token length stats (text) ---")
lengths = [len(tokenizer.encode(ex["text"])) for ex in formatted]
print(f"min={min(lengths)} median={sorted(lengths)[len(lengths)//2]} max={max(lengths)}")


In [9]:
# 冒烟：未微调的基础模型对同一 prompt 随意续写（对比 SFT 前后用）
model.eval()
prompt = formatted[0]["prompt"]
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
print("=== base GPT-2 completion (before SFT) ===")
print(tokenizer.decode(out[0], skip_special_tokens=True))


=== base GPT-2 completion (before SFT) ===
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain what supervised fine-tuning is in one sentence.

### Input:


### Response:
Fine-tuning computers.


## 1. 监督微调

要点（与笔记一致）：
- 低学习率（约 `2e-5`），避免灾难性遗忘
- 短训练（这里 2 个 epoch）
- **只对 Response 词元算 loss**（prompt 部分 mask 掉）


In [6]:
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 256
BATCH_SIZE = 4
EPOCHS = 2
LR = 2e-5
SFT_OUT = ROOT / "models" / "gpt2-sft"


class AlpacaSFTDataset(Dataset):
    """tokenize prompt+response，并对 prompt 段做 labels=-100（不算 loss）。"""

    def __init__(self, examples, tokenizer, max_len=MAX_LEN):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        prompt_ids = self.tokenizer.encode(ex["prompt"], add_special_tokens=False)
        full_ids = self.tokenizer.encode(ex["text"], add_special_tokens=False)

        # 截断（保留尾部 response 优先：先截 prompt 过长的情况很少见，这里简单截尾）
        full_ids = full_ids[: self.max_len]
        prompt_len = min(len(prompt_ids), len(full_ids))

        labels = full_ids.copy()
        # causal LM：通常 labels 与 input_ids 对齐，HF 内部会 shift
        # prompt 位置不计入 loss
        labels[:prompt_len] = [-100] * prompt_len

        attn = [1] * len(full_ids)
        return {
            "input_ids": full_ids,
            "attention_mask": attn,
            "labels": labels,
            "prompt_len": prompt_len,
        }


def collate_sft(batch):
    pad_id = tokenizer.pad_token_id
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        L = len(x["input_ids"])
        pad_n = max_len - L
        input_ids.append(x["input_ids"] + [pad_id] * pad_n)
        attention_mask.append(x["attention_mask"] + [0] * pad_n)
        # pad 位置 labels 也必须是 -100
        labels.append(x["labels"] + [-100] * pad_n)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


# 简单划分：后 10 条做 val
train_examples = formatted[:-10]
val_examples = formatted[-10:]
train_ds = AlpacaSFTDataset(train_examples, tokenizer)
val_ds = AlpacaSFTDataset(val_examples, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_sft)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_sft)

# 看一条 mask 是否正确
sample = train_ds[0]
n_resp = sum(1 for y in sample["labels"] if y != -100)
print(f"train={len(train_ds)} val={len(val_ds)}")
print(f"example0: total_tokens={len(sample['input_ids'])} prompt_masked={sample['prompt_len']} response_tokens={n_resp}")


train=90 val=10
example0: total_tokens=78 prompt_masked=48 response_tokens=30


In [7]:
from torch.optim import AdamW

model.train()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)


def eval_loss(loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            total += out.loss.item()
            n += 1
    model.train()
    return total / max(n, 1)


print(f"SFT start | epochs={EPOCHS} lr={LR} batch={BATCH_SIZE} device={DEVICE}")
history = []
global_step = 0
for epoch in range(1, EPOCHS + 1):
    running = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running += loss.item()
        global_step += 1
        if step % 5 == 0 or step == len(train_loader):
            avg = running / step
            print(f"epoch {epoch} step {step}/{len(train_loader)} train_loss={avg:.4f}")

    vloss = eval_loss(val_loader)
    history.append({"epoch": epoch, "train_loss": running / len(train_loader), "val_loss": vloss})
    print(f"epoch {epoch} done | val_loss={vloss:.4f}")

SFT_OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_OUT)
tokenizer.save_pretrained(SFT_OUT)
print("saved SFT model ->", SFT_OUT)
print("history:", history)


SFT start | epochs=2 lr=2e-05 batch=4 device=mps


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


epoch 1 step 5/23 train_loss=5.0216
epoch 1 step 10/23 train_loss=4.5375
epoch 1 step 15/23 train_loss=4.2171
epoch 1 step 20/23 train_loss=4.0108
epoch 1 step 23/23 train_loss=4.0085
epoch 1 done | val_loss=4.2236
epoch 2 step 5/23 train_loss=2.9514
epoch 2 step 10/23 train_loss=2.6618
epoch 2 step 15/23 train_loss=2.9265
epoch 2 step 20/23 train_loss=2.8813
epoch 2 step 23/23 train_loss=2.9523
epoch 2 done | val_loss=4.1284


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved SFT model -> /Users/keyficller/Documents/AIEngineering/sources/10_LLMs_FROM_SCRATCH/06_Instruction_Tuning/models/gpt2-sft
history: [{'epoch': 1, 'train_loss': 4.008516912874968, 'val_loss': 4.223612944285075}, {'epoch': 2, 'train_loss': 2.9522626399993896, 'val_loss': 4.128433704376221}]


## 2. 微调前后对比

同一条 instruction，看 base GPT-2 vs SFT 后的续写。


In [ ]:
model.eval()
prompt = formatted[0]["prompt"]
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
print("=== base GPT-2 completion (before SFT) ===")
print(tokenizer.decode(out[0], skip_special_tokens=True))

=== after SFT ===

[0] gold: Supervised fine-tuning trains a pretrained language model on instruction-response pairs so it learns to follow instructions rather than only continue text.
[0] pred: Fine-tuning is the process of using a series of rules to determine what a rule should be. It consists of an algorithm that randomly selects a rule for a given set of rules to be well-tuned toward.

[1] gold: def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)
[1] pred: n = 1

[2] gold: HyperText Transfer Protocol
[2] pred: HTTP
